In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)

# Load league-wide game data and filter to Arizona home games
data = pd.read_csv('../../data/league_weather_2021_2025.csv')
az = data[data['home_team'] == 'AZ'].copy()

az['game_date'] = pd.to_datetime(az['game_date'])
az['month'] = az['game_date'].dt.month
month_map = {3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct'}
az['month_name'] = az['month'].map(month_map)

az['temp_bin'] = pd.qcut(az['temp_f'], q=5, duplicates='drop')
az['pres_bin'] = pd.qcut(az['pres'], q=5, duplicates='drop')

print(f'Total AZ home games: {len(az)}')
print('\nTemperature bins and game counts:')
print(az['temp_bin'].value_counts().sort_index())
print('\nPressure bins and game counts:')
print(az['pres_bin'].value_counts().sort_index())

In [ ]:
# Temperature Quintiles: Summary Table
temp_summary = az.groupby('temp_bin', observed=True).agg(
    games=('temp_f', 'size'),
    temp_mean=('temp_f', 'mean'),
    temp_std=('temp_f', 'std'),
    total_runs_mean=('total_runs', 'mean'),
    home_runs_mean=('home_runs_scored', 'mean'),
    away_runs_mean=('away_runs_scored', 'mean'),
    strikeouts_mean=('strikeouts', 'mean'),
    home_runs_hit_mean=('home_runs_hit', 'mean'),
    hr_h_ratio_mean=('hr_h_ratio', 'mean'),
    pres_mean=('pres', 'mean'),
    rhum_mean=('rhum', 'mean'),
    wspd_mean=('wspd_mph', 'mean'),
).round(2)

temp_summary.index.name = 'Temperature Quintile (degF)'
temp_summary.columns = [
    'Games', 'Temp Mean', 'Temp Std', 'Total Runs Mean', 'Home Runs Mean',
    'Away Runs Mean', 'Strikeouts Mean', 'Home Runs Hit Mean', 'HR:H Ratio Mean',
    'Pressure Mean', 'Humidity Mean', 'Wind Mean'
]
temp_summary

In [ ]:
# Temperature Quintiles: Scoring and Contact Profile
temp_order = az['temp_bin'].cat.categories
temp_labels = [str(b).replace('(', '').replace(']', '').replace(', ', ' to ') for b in temp_order]
x = np.arange(len(temp_order))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].bar(x, az.groupby('temp_bin', observed=True)['total_runs'].mean().reindex(temp_order), color='#d62728', alpha=0.8, edgecolor='black', linewidth=0.5)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(temp_labels, rotation=20, ha='right')
axes[0, 0].set_ylabel('Total Runs')
axes[0, 0].set_title('Total Runs by Temperature Quintile')
axes[0, 0].yaxis.grid(True, alpha=0.3)

bar_width = 0.35
home_means = az.groupby('temp_bin', observed=True)['home_runs_scored'].mean().reindex(temp_order)
away_means = az.groupby('temp_bin', observed=True)['away_runs_scored'].mean().reindex(temp_order)
axes[0, 1].bar(x - bar_width / 2, home_means, width=bar_width, color='#1f77b4', alpha=0.8, label='Home Runs Scored', edgecolor='black', linewidth=0.5)
axes[0, 1].bar(x + bar_width / 2, away_means, width=bar_width, color='#ff7f0e', alpha=0.8, label='Away Runs Scored', edgecolor='black', linewidth=0.5)
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(temp_labels, rotation=20, ha='right')
axes[0, 1].set_ylabel('Runs')
axes[0, 1].set_title('Home vs Away Runs by Temperature Quintile')
axes[0, 1].legend()
axes[0, 1].yaxis.grid(True, alpha=0.3)

axes[1, 0].plot(x, az.groupby('temp_bin', observed=True)['home_runs_hit'].mean().reindex(temp_order), marker='o', color='#9467bd', linewidth=2)
axes[1, 0].plot(x, az.groupby('temp_bin', observed=True)['hr_h_ratio'].mean().reindex(temp_order), marker='o', color='#2ca02c', linewidth=2)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(temp_labels, rotation=20, ha='right')
axes[1, 0].set_title('Home Runs and HR:H Ratio by Temperature Quintile')
axes[1, 0].set_ylabel('Home Runs / HR:H Ratio')
axes[1, 0].legend(['Home Runs Hit', 'HR:H Ratio'])
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(x, az.groupby('temp_bin', observed=True)['pres'].mean().reindex(temp_order), marker='o', color='#9467bd', linewidth=2)
axes[1, 1].plot(x, az.groupby('temp_bin', observed=True)['rhum'].mean().reindex(temp_order), marker='o', color='#1f77b4', linewidth=2)
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(temp_labels, rotation=20, ha='right')
axes[1, 1].set_title('Pressure and Humidity by Temperature Quintile')
axes[1, 1].set_ylabel('Pressure / Humidity')
axes[1, 1].legend(['Pressure', 'Humidity'])
axes[1, 1].grid(True, alpha=0.3)

fig.suptitle('Chase Field: Temperature Quintiles and Scoring Profile', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Pressure Quintiles: Summary Table
pres_summary = az.groupby('pres_bin', observed=True).agg(
    games=('pres', 'size'),
    pres_mean=('pres', 'mean'),
    pres_std=('pres', 'std'),
    total_runs_mean=('total_runs', 'mean'),
    away_runs_mean=('away_runs_scored', 'mean'),
    strikeouts_mean=('strikeouts', 'mean'),
    home_runs_hit_mean=('home_runs_hit', 'mean'),
    hr_h_ratio_mean=('hr_h_ratio', 'mean'),
    temp_mean=('temp_f', 'mean'),
    rhum_mean=('rhum', 'mean'),
    wspd_mean=('wspd_mph', 'mean'),
).round(2)

pres_summary.index.name = 'Pressure Quintile (hPa)'
pres_summary.columns = [
    'Games', 'Pressure Mean', 'Pressure Std', 'Total Runs Mean', 'Away Runs Mean',
    'Strikeouts Mean', 'Home Runs Hit Mean', 'HR:H Ratio Mean',
    'Temp Mean', 'Humidity Mean', 'Wind Mean'
]
pres_summary

In [ ]:
# Pressure Quintiles: Runs, Strikeouts, and Conditions
pres_order = az['pres_bin'].cat.categories
pres_labels = [str(b).replace('(', '').replace(']', '').replace(', ', ' to ') for b in pres_order]
x = np.arange(len(pres_order))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].bar(x, az.groupby('pres_bin', observed=True)['away_runs_scored'].mean().reindex(pres_order), color='#ff7f0e', alpha=0.8, edgecolor='black', linewidth=0.5)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(pres_labels, rotation=20, ha='right')
axes[0, 0].set_ylabel('Away Runs')
axes[0, 0].set_title('Away Runs by Pressure Quintile')
axes[0, 0].yaxis.grid(True, alpha=0.3)

axes[0, 1].bar(x, az.groupby('pres_bin', observed=True)['strikeouts'].mean().reindex(pres_order), color='#1f77b4', alpha=0.8, edgecolor='black', linewidth=0.5)
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(pres_labels, rotation=20, ha='right')
axes[0, 1].set_ylabel('Strikeouts')
axes[0, 1].set_title('Strikeouts by Pressure Quintile')
axes[0, 1].yaxis.grid(True, alpha=0.3)

axes[1, 0].plot(x, az.groupby('pres_bin', observed=True)['temp_f'].mean().reindex(pres_order), marker='o', color='#d62728', linewidth=2)
axes[1, 0].plot(x, az.groupby('pres_bin', observed=True)['rhum'].mean().reindex(pres_order), marker='o', color='#1f77b4', linewidth=2)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(pres_labels, rotation=20, ha='right')
axes[1, 0].set_title('Temperature and Humidity by Pressure Quintile')
axes[1, 0].set_ylabel('Temp / Humidity')
axes[1, 0].legend(['Temperature', 'Humidity'])
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(x, az.groupby('pres_bin', observed=True)['wspd_mph'].mean().reindex(pres_order), marker='o', color='#2ca02c', linewidth=2)
axes[1, 1].plot(x, az.groupby('pres_bin', observed=True)['hr_h_ratio'].mean().reindex(pres_order), marker='o', color='#9467bd', linewidth=2)
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(pres_labels, rotation=20, ha='right')
axes[1, 1].set_title('Wind and HR:H Ratio by Pressure Quintile')
axes[1, 1].set_ylabel('Wind / HR:H Ratio')
axes[1, 1].legend(['Wind Speed', 'HR:H Ratio'])
axes[1, 1].grid(True, alpha=0.3)

fig.suptitle('Chase Field: Pressure Quintiles and Offensive Outcomes', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Temperature vs Pressure: Scatter Diagnostics
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
scatter = ax.scatter(az['temp_f'], az['pres'], c=az['total_runs'], cmap='YlOrRd', alpha=0.65, edgecolors='black', linewidth=0.3, s=45)
ax.set_xlabel('Temperature (degF)')
ax.set_ylabel('Pressure (hPa)')
ax.set_title('Temperature vs Pressure, Colored by Total Runs')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Total Runs')
ax.grid(True, alpha=0.3)

ax = axes[1]
scatter = ax.scatter(az['temp_f'], az['pres'], c=az['strikeouts'], cmap='Blues', alpha=0.65, edgecolors='black', linewidth=0.3, s=45)
ax.set_xlabel('Temperature (degF)')
ax.set_ylabel('Pressure (hPa)')
ax.set_title('Temperature vs Pressure, Colored by Strikeouts')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Strikeouts')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Interaction View: Temperature x Pressure Quintiles
runs_pivot = az.pivot_table(index='temp_bin', columns='pres_bin', values='total_runs', aggfunc='mean', observed=True)
k_pivot = az.pivot_table(index='temp_bin', columns='pres_bin', values='strikeouts', aggfunc='mean', observed=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

im1 = axes[0].imshow(runs_pivot.values, cmap='YlOrRd', aspect='auto')
axes[0].set_xticks(np.arange(runs_pivot.shape[1]))
axes[0].set_yticks(np.arange(runs_pivot.shape[0]))
axes[0].set_xticklabels([str(x).replace('(', '').replace(']', '').replace(', ', ' to ') for x in runs_pivot.columns], rotation=25, ha='right')
axes[0].set_yticklabels([str(x).replace('(', '').replace(']', '').replace(', ', ' to ') for x in runs_pivot.index])
axes[0].set_xlabel('Pressure Quintile')
axes[0].set_ylabel('Temperature Quintile')
axes[0].set_title('Mean Total Runs')
for i in range(runs_pivot.shape[0]):
    for j in range(runs_pivot.shape[1]):
        val = runs_pivot.iloc[i, j]
        if pd.notna(val):
            axes[0].text(j, i, f'{val:.2f}', ha='center', va='center', color='black', fontsize=9)
plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

im2 = axes[1].imshow(k_pivot.values, cmap='Blues', aspect='auto')
axes[1].set_xticks(np.arange(k_pivot.shape[1]))
axes[1].set_yticks(np.arange(k_pivot.shape[0]))
axes[1].set_xticklabels([str(x).replace('(', '').replace(']', '').replace(', ', ' to ') for x in k_pivot.columns], rotation=25, ha='right')
axes[1].set_yticklabels([str(x).replace('(', '').replace(']', '').replace(', ', ' to ') for x in k_pivot.index])
axes[1].set_xlabel('Pressure Quintile')
axes[1].set_ylabel('Temperature Quintile')
axes[1].set_title('Mean Strikeouts')
for i in range(k_pivot.shape[0]):
    for j in range(k_pivot.shape[1]):
        val = k_pivot.iloc[i, j]
        if pd.notna(val):
            axes[1].text(j, i, f'{val:.2f}', ha='center', va='center', color='black', fontsize=9)
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

fig.suptitle('Chase Field: Temperature–Pressure Interaction View', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()